# Stage 3: DistilBERT Fine-Tuning

[1] Restoring Environment:

In [1]:
!pip install transformers datasets scikit-learn groq pandas numpy torch -q

from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 12.4 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import torch

pseudo_df = pd.read_csv('/content/drive/MyDrive/KD_Project/pseudo_labels_final.csv')
val_df    = pd.read_csv('/content/drive/MyDrive/KD_Project/val.csv')
test_df   = pd.read_csv('/content/drive/MyDrive/KD_Project/test.csv')

print(f"Pseudo-labelled train : {len(pseudo_df)}")
print(f"Validation            : {len(val_df)}")
print(f"Test                  : {len(test_df)}")
print(f"GPU available         : {torch.cuda.is_available()}")
print(f"GPU name              : {torch.cuda.get_device_name(0)}")

Pseudo-labelled train : 3876
Validation            : 485
Test                  : 485
GPU available         : True
GPU name              : Tesla T4


[2] Defining Label Maps + Tokenizer:

In [3]:
from transformers import DistilBertTokenizerFast

# Label mappings
SENTIMENT_MAP = {"negative": 0, "neutral": 1, "positive": 2}
URGENCY_MAP   = {"non-urgent": 0, "urgent": 1}

# Reverse maps for readable output later
INV_SENTIMENT = {v: k for k, v in SENTIMENT_MAP.items()}
INV_URGENCY   = {v: k for k, v in URGENCY_MAP.items()}

# Load DistilBERT tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# Apply label maps to pseudo-labelled training data
pseudo_df["sentiment_id"] = pseudo_df["pseudo_sentiment"].map(SENTIMENT_MAP)
pseudo_df["urgency_id"]   = pseudo_df["pseudo_urgency"].map(URGENCY_MAP)

# Apply label maps to validation data
# Val uses gold sentiment labels; urgency not in val — we'll predict only
val_df["sentiment_id"] = val_df["sentiment"].map(SENTIMENT_MAP)

# Drop any rows where mapping failed
pseudo_df = pseudo_df.dropna(subset=["sentiment_id", "urgency_id"]).reset_index(drop=True)
pseudo_df["sentiment_id"] = pseudo_df["sentiment_id"].astype(int)
pseudo_df["urgency_id"]   = pseudo_df["urgency_id"].astype(int)

print(f"Training samples ready : {len(pseudo_df)}")
print(f"\nSentiment label distribution:")
print(pseudo_df["sentiment_id"].value_counts().sort_index().rename(INV_SENTIMENT))
print(f"\nUrgency label distribution:")
print(pseudo_df["urgency_id"].value_counts().sort_index().rename(INV_URGENCY))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Training samples ready : 3876

Sentiment label distribution:
sentiment_id
negative     500
neutral     2495
positive     881
Name: count, dtype: int64

Urgency label distribution:
urgency_id
non-urgent    3747
urgent         129
Name: count, dtype: int64


[3] BuildING PyTorch Dataset:

In [4]:
from torch.utils.data import Dataset, DataLoader

class FinancialDataset(Dataset):
    def __init__(self, texts, sentiment_labels, urgency_labels=None,
                 tokenizer=tokenizer, max_length=128):
        self.encodings        = tokenizer(
            list(texts),
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors='pt'
        )
        self.sentiment_labels = torch.tensor(sentiment_labels, dtype=torch.long)
        self.urgency_labels   = torch.tensor(urgency_labels, dtype=torch.long) \
                                if urgency_labels is not None else None

    def __len__(self):
        return len(self.sentiment_labels)

    def __getitem__(self, idx):
        item = {
            'input_ids'      : self.encodings['input_ids'][idx],
            'attention_mask' : self.encodings['attention_mask'][idx],
            'sentiment_label': self.sentiment_labels[idx],
        }
        if self.urgency_labels is not None:
            item['urgency_label'] = self.urgency_labels[idx]
        return item


# Build train dataset
train_dataset = FinancialDataset(
    texts            = pseudo_df["text"].values,
    sentiment_labels = pseudo_df["sentiment_id"].values,
    urgency_labels   = pseudo_df["urgency_id"].values
)

# Build val dataset — no urgency labels available, use dummy zeros
val_dataset = FinancialDataset(
    texts            = val_df["text"].values,
    sentiment_labels = val_df["sentiment_id"].values,
    urgency_labels   = np.zeros(len(val_df), dtype=int)
)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Dataset built successfully.")

Train batches : 243
Val batches   : 16
Dataset built successfully.


[4] Computing Class Weights:

In [5]:
from sklearn.utils.class_weight import compute_class_weight

# Sentiment class weights
sentiment_weights = compute_class_weight(
    class_weight = 'balanced',
    classes      = np.array([0, 1, 2]),
    y            = pseudo_df["sentiment_id"].values
)

# Urgency class weights — manual due to extreme 29:1 imbalance
urgency_weights = compute_class_weight(
    class_weight = 'balanced',
    classes      = np.array([0, 1]),
    y            = pseudo_df["urgency_id"].values
)

# Move to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

sentiment_weights_tensor = torch.tensor(sentiment_weights, dtype=torch.float).to(device)
urgency_weights_tensor   = torch.tensor(urgency_weights,   dtype=torch.float).to(device)

print(f"Device : {device}")
print(f"\nSentiment class weights:")
for i, w in enumerate(sentiment_weights):
    print(f"  {INV_SENTIMENT[i]:>12} (class {i}) : {w:.4f}")

print(f"\nUrgency class weights:")
for i, w in enumerate(urgency_weights):
    print(f"  {INV_URGENCY[i]:>12} (class {i}) : {w:.4f}")

Device : cuda

Sentiment class weights:
      negative (class 0) : 2.5840
       neutral (class 1) : 0.5178
      positive (class 2) : 1.4665

Urgency class weights:
    non-urgent (class 0) : 0.5172
        urgent (class 1) : 15.0233


[5] Dual-Head DistilBERT Mode:

In [6]:
import torch.nn as nn
from transformers import DistilBertModel

class DualHeadDistilBERT(nn.Module):
    """
    DistilBERT with two independent classification heads:
    - Head 1: Sentiment (negative / neutral / positive)
    - Head 2: Urgency   (non-urgent / urgent)
    Both share the same DistilBERT encoder (CLS token representation).
    """
    def __init__(self, num_sentiment=3, num_urgency=2, dropout=0.3):
        super(DualHeadDistilBERT, self).__init__()

        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout    = nn.Dropout(dropout)

        # Independent classification heads
        self.sentiment_head = nn.Linear(768, num_sentiment)
        self.urgency_head   = nn.Linear(768, num_urgency)

    def forward(self, input_ids, attention_mask):
        # Get DistilBERT contextual embeddings
        outputs    = self.distilbert(
            input_ids      = input_ids,
            attention_mask = attention_mask
        )

        # CLS token = sentence-level representation (position 0)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)

        # Dual head predictions
        sentiment_logits = self.sentiment_head(cls_output)
        urgency_logits   = self.urgency_head(cls_output)

        return sentiment_logits, urgency_logits


# Initialise and move to GPU
model = DualHeadDistilBERT().to(device)

# Count parameters
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model loaded on : {device}")
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on : cuda
Total parameters    : 66,366,725
Trainable parameters: 66,366,725


[6] Training Setup:

In [7]:
from transformers import get_linear_schedule_with_warmup

# Loss functions with class weights
sentiment_criterion = nn.CrossEntropyLoss(weight=sentiment_weights_tensor)
urgency_criterion   = nn.CrossEntropyLoss(weight=urgency_weights_tensor)

# Task loss weights — sentiment is primary task
ALPHA = 0.6   # sentiment weight
BETA  = 0.4   # urgency weight

# Optimizer — AdamW with weight decay
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr           = 2e-5,
    weight_decay = 0.01
)

# Training config
EPOCHS     = 5
TOTAL_STEPS = len(train_loader) * EPOCHS
WARMUP_STEPS = int(0.1 * TOTAL_STEPS)   # 10% warmup

# Linear warmup + decay scheduler
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = WARMUP_STEPS,
    num_training_steps = TOTAL_STEPS
)

print(f"Epochs          : {EPOCHS}")
print(f"Total steps     : {TOTAL_STEPS}")
print(f"Warmup steps    : {WARMUP_STEPS}")
print(f"Batch size      : 16")
print(f"Learning rate   : 2e-5")
print(f"Loss weights    : α={ALPHA} (sentiment) + β={BETA} (urgency)")
print(f"Training setup complete.")

Epochs          : 5
Total steps     : 1215
Warmup steps    : 121
Batch size      : 16
Learning rate   : 2e-5
Loss weights    : α=0.6 (sentiment) + β=0.4 (urgency)
Training setup complete.


[7] Training Loop:

In [8]:
from sklearn.metrics import f1_score
import time

best_val_f1   = 0.0
best_model_state = None
train_history = []

print("Starting training...\n")
print(f"{'Epoch':<8}{'Train Loss':<14}{'Val Loss':<12}{'Val F1 (Macro)':<18}{'Time'}")
print("-" * 60)

for epoch in range(EPOCHS):
    epoch_start = time.time()

    # -------- TRAINING --------
    model.train()
    total_train_loss = 0

    for batch in train_loader:
        input_ids       = batch['input_ids'].to(device)
        attention_mask  = batch['attention_mask'].to(device)
        sentiment_labels = batch['sentiment_label'].to(device)
        urgency_labels  = batch['urgency_label'].to(device)

        optimizer.zero_grad()

        sentiment_logits, urgency_logits = model(input_ids, attention_mask)

        # Weighted combined loss
        loss_sentiment = sentiment_criterion(sentiment_logits, sentiment_labels)
        loss_urgency   = urgency_criterion(urgency_logits, urgency_labels)
        loss           = ALPHA * loss_sentiment + BETA * loss_urgency

        loss.backward()

        # Gradient clipping — prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # -------- VALIDATION --------
    model.eval()
    total_val_loss = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids        = batch['input_ids'].to(device)
            attention_mask   = batch['attention_mask'].to(device)
            sentiment_labels = batch['sentiment_label'].to(device)
            urgency_labels   = batch['urgency_label'].to(device)

            sentiment_logits, urgency_logits = model(input_ids, attention_mask)

            loss_sentiment = sentiment_criterion(sentiment_logits, sentiment_labels)
            loss_urgency   = urgency_criterion(urgency_logits, urgency_labels)
            val_loss       = ALPHA * loss_sentiment + BETA * loss_urgency

            total_val_loss += val_loss.item()

            preds  = torch.argmax(sentiment_logits, dim=1).cpu().numpy()
            labels = sentiment_labels.cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels)

    avg_val_loss = total_val_loss / len(val_loader)
    val_f1       = f1_score(all_labels, all_preds, average='macro')

    epoch_time = time.time() - epoch_start

    print(f"Epoch {epoch+1:<4} {avg_train_loss:<14.4f}{avg_val_loss:<12.4f}{val_f1:<18.4f}{epoch_time:.1f}s")

    train_history.append({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
        "val_f1": val_f1
    })

    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1      = val_f1
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        print(f"         → New best model saved (Val F1: {val_f1:.4f})")

print(f"\nTraining complete. Best Val Macro F1 : {best_val_f1:.4f}")

Starting training...

Epoch   Train Loss    Val Loss    Val F1 (Macro)    Time
------------------------------------------------------------
Epoch 1    0.6763        0.4835      0.7517            44.9s
         → New best model saved (Val F1: 0.7517)
Epoch 2    0.3110        0.3649      0.7983            46.6s
         → New best model saved (Val F1: 0.7983)
Epoch 3    0.1737        0.5253      0.8023            49.3s
         → New best model saved (Val F1: 0.8023)
Epoch 4    0.0966        0.5814      0.8004            48.6s
Epoch 5    0.0635        0.6535      0.8028            49.3s
         → New best model saved (Val F1: 0.8028)

Training complete. Best Val Macro F1 : 0.8028


[8] Full Evaluation + Save Model

In [9]:
from sklearn.metrics import classification_report

# Load best model weights
model.load_state_dict(best_model_state)
model.eval()

# Full evaluation on validation set
all_sentiment_preds  = []
all_sentiment_labels = []
all_urgency_preds    = []

with torch.no_grad():
    for batch in val_loader:
        input_ids        = batch['input_ids'].to(device)
        attention_mask   = batch['attention_mask'].to(device)

        sentiment_logits, urgency_logits = model(input_ids, attention_mask)

        sentiment_preds = torch.argmax(sentiment_logits, dim=1).cpu().numpy()
        urgency_preds   = torch.argmax(urgency_logits,   dim=1).cpu().numpy()

        all_sentiment_preds.extend(sentiment_preds)
        all_sentiment_labels.extend(batch['sentiment_label'].numpy())
        all_urgency_preds.extend(urgency_preds)

# Sentiment report
print("=" * 55)
print("SENTIMENT CLASSIFICATION REPORT (Validation Set)")
print("=" * 55)
print(classification_report(
    all_sentiment_labels,
    all_sentiment_preds,
    target_names=["negative", "neutral", "positive"]
))

# Urgency distribution
print("=" * 55)
print("URGENCY PREDICTIONS (Validation Set)")
print("=" * 55)
from collections import Counter
urgency_counts = Counter(all_urgency_preds)
print(f"non-urgent : {urgency_counts[0]}")
print(f"urgent     : {urgency_counts[1]}")

# Save model + tokenizer to Drive
import os
SAVE_DIR = '/content/drive/MyDrive/KD_Project/distilbert_finetuned'
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save(best_model_state, f'{SAVE_DIR}/model_weights.pt')
tokenizer.save_pretrained(SAVE_DIR)

# Save training history
pd.DataFrame(train_history).to_csv(f'{SAVE_DIR}/training_history.csv', index=False)

print(f"\nModel saved to Drive at:")
print(f"  {SAVE_DIR}/model_weights.pt")
print(f"  {SAVE_DIR}/tokenizer files")
print(f"  {SAVE_DIR}/training_history.csv")

SENTIMENT CLASSIFICATION REPORT (Validation Set)
              precision    recall  f1-score   support

    negative       0.80      0.98      0.88        60
     neutral       0.83      0.86      0.84       288
    positive       0.76      0.62      0.68       137

    accuracy                           0.81       485
   macro avg       0.80      0.82      0.80       485
weighted avg       0.81      0.81      0.80       485

URGENCY PREDICTIONS (Validation Set)
non-urgent : 468
urgent     : 17

Model saved to Drive at:
  /content/drive/MyDrive/KD_Project/distilbert_finetuned/model_weights.pt
  /content/drive/MyDrive/KD_Project/distilbert_finetuned/tokenizer files
  /content/drive/MyDrive/KD_Project/distilbert_finetuned/training_history.csv
